In [ ]:
import numpy as np
import pandas as pd

def calculate_pv_output(solar_radiation, temperature,
                       C_R_PV=1000, G_T_STC=1000, T_C_STC=25, alpha_p=-0.004):
    """
    Calculate photovoltaic power output based on weather parameters.

    Parameters:
    -----------
    solar_radiation : float or array-like
        Solar radiation incident on PV panel surface (W/m²)
    temperature : float or array-like
        Cell/ambient temperature (°C)
    C_R_PV : float, default=1000
        Rated capacity of PV array under standard test conditions (W)
    G_T_STC : float, default=1000
        Solar radiation under standard test conditions (W/m²)
    T_C_STC : float, default=25
        Cell temperature under standard test conditions (°C)
    alpha_p : float, default=-0.004
        Temperature coefficient of power (%/°C), expressed as decimal (-0.4% = -0.004)

    Returns:
    --------
    float or array-like
        Predicted photovoltaic power output (W)

    Formula:
    --------
    P_out_pv = C_R_PV * (G_T / G_T_STC) * [1 + alpha_p * (T_C - T_C_STC)]
    """

    # Calculate radiation ratio
    radiation_ratio = solar_radiation / G_T_STC

    # Calculate temperature effect
    temperature_effect = 1 + alpha_p * (temperature - T_C_STC)

    # Calculate PV output
    P_out_pv = C_R_PV * radiation_ratio * temperature_effect

    # Ensure non-negative output (no power generation when no radiation)
    P_out_pv = np.maximum(P_out_pv, 0)

    return P_out_pv


# Example usage with single values
def example_single_calculation():
    """Example with single weather reading"""
    solar_rad = 800  # W/m²
    temp = 30  # °C

    output = calculate_pv_output(solar_rad, temp)
    print(f"Solar Radiation: {solar_rad} W/m²")
    print(f"Temperature: {temp} °C")
    print(f"PV Output: {output:.2f} W")
    print()


# Example usage with pandas DataFrame
def example_dataframe_calculation():
    """Example with DataFrame - similar to the IMS API data"""

    # Sample weather data
    data = {
        'timestamp': pd.date_range('2024-01-01', periods=24, freq='h'),
        'solar_radiation': [0, 0, 0, 0, 0, 50, 200, 400, 600, 800, 900, 950,
                           950, 900, 800, 600, 400, 200, 50, 0, 0, 0, 0, 0],
        'temperature': [15, 14, 14, 13, 13, 14, 16, 19, 22, 25, 28, 30,
                       31, 30, 29, 27, 24, 20, 18, 16, 15, 15, 14, 14]
    }

    df = pd.DataFrame(data)

    # Calculate PV output for entire dataset
    df['pv_output'] = calculate_pv_output(
        df['solar_radiation'],
        df['temperature']
    )

    print("Sample Data with PV Output:")
    print(df.head(10))
    print()
    print(f"Total Daily Energy: {df['pv_output'].sum():.2f} Wh")
    print(f"Peak Power: {df['pv_output'].max():.2f} W")
    print(f"Average Power (daylight hours): {df[df['solar_radiation'] > 0]['pv_output'].mean():.2f} W")

    return df


# Process IMS API data with date_time column
def process_ims_data(df):
    """
    Process IMS API weather data and add PV output labels

    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing 'date_time', 'solar_radiation' and 'temperature' columns

    Returns:
    --------
    pandas.DataFrame
        Original DataFrame with added 'pv_output' column and properly formatted date_time
    """

    # Check required columns exist
    required_cols = ['date_time', 'solar_radiation', 'temperature']
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Convert date_time to datetime if it's not already
    if not pd.api.types.is_datetime64_any_dtype(df['date_time']):
        df['date_time'] = pd.to_datetime(df['date_time'])

    # Sort by date_time to ensure chronological order
    df = df.sort_values('date_time').reset_index(drop=True)

    # Handle missing values
    df['solar_radiation'] = df['solar_radiation'].fillna(0)
    df['temperature'] = df['temperature'].fillna(df['temperature'].mean())

    # Calculate PV output (true labels)
    df['pv_output'] = calculate_pv_output(
        df['solar_radiation'].values,
        df['temperature'].values
    )

    print(f"Processed {len(df)} records")
    print(f"Date range: {df['date_time'].min()} to {df['date_time'].max()}")
    print(f"PV Output range: {df['pv_output'].min():.2f} W to {df['pv_output'].max():.2f} W")

    return df


# Run examples
if __name__ == "__main__":
    print("="*60)
    print("Single Value Example:")
    print("="*60)
    example_single_calculation()

    print("="*60)
    print("DataFrame Example:")
    print("="*60)
    df_example = example_dataframe_calculation()

    print("\n" + "="*60)
    print("CSV File Example:")
    print("="*60)

    # Example: Reading from CSV and calculating true labels
    try:
        csv_file_path = '/content/data.csv'

        print(f"Reading CSV file: {csv_file_path}")
        df = pd.read_csv(csv_file_path)

        print("\nOriginal DataFrame:")
        print(df.head())
        print(f"\nOriginal columns: {df.columns.tolist()}")

        # Process the data and calculate PV output (true labels)
        print("\nProcessing IMS data and calculating true labels...")
        df_with_labels = process_ims_data(df)

        print("\nDataFrame with True Labels:")
        print(df_with_labels[['date_time', 'solar_radiation', 'temperature', 'pv_output']].head(10))

        # Save the DataFrame with true labels to a new CSV
        output_file = 'weather_data_with_labels.csv'
        df_with_labels.to_csv(output_file, index=False)
        print(f"\nSaved processed data with true labels to: {output_file}")

    except FileNotFoundError:
        print(f"CSV file '{csv_file_path}' not found.")
        print("Please update the csv_file_path variable with your actual file path.")
        print("\nExpected CSV format:")
        print("date_time,solar_radiation,temperature")
        print("2024-01-01 00:00:00,0,15")
        print("2024-01-01 01:00:00,0,14")
        print("...")
    except Exception as e:
        print(f"Error processing CSV: {str(e)}")

Single Value Example:
Solar Radiation: 800 W/m²
Temperature: 30 °C
PV Output: 784.00 W

DataFrame Example:
Sample Data with PV Output:
            timestamp  solar_radiation  temperature  pv_output
0 2024-01-01 00:00:00                0           15        0.0
1 2024-01-01 01:00:00                0           14        0.0
2 2024-01-01 02:00:00                0           14        0.0
3 2024-01-01 03:00:00                0           13        0.0
4 2024-01-01 04:00:00                0           13        0.0
5 2024-01-01 05:00:00               50           14       52.2
6 2024-01-01 06:00:00              200           16      207.2
7 2024-01-01 07:00:00              400           19      409.6
8 2024-01-01 08:00:00              600           22      607.2
9 2024-01-01 09:00:00              800           25      800.0

Total Daily Energy: 7745.00 Wh
Peak Power: 931.00 W
Average Power (daylight hours): 553.21 W

CSV File Example:
Reading CSV file: /content/test.csv

Original DataFrame:
  

/tmp/ipython-input-1714172708.py:68: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  'timestamp': pd.date_range('2024-01-01', periods=24, freq='H'),
